In [1]:
import base64
import json
import mimetypes
import re
from pathlib import Path
from openai import OpenAI

# 1. Configure GB10 Endpoint (use your GB10 IP: 10.184.135.111 or shared server 10.246.100.230)
GB10_BASE_URL = "http://10.246.100.230/v1"
GB10_API_KEY = "usaadatathon"
GB10_MODEL = "qwen3-vl-8b-instruct"

client = OpenAI(base_url=GB10_BASE_URL, api_key=GB10_API_KEY, timeout=120)

# 2. Quick Connection Sanity Check ("Hello World")
completion = client.chat.completions.create(
    model=GB10_MODEL,
    messages=[
        {"role": "system", "content": "You are a friendly AI that loves meeting new people."},
        {"role": "user", "content": "Write a short but fun introduction for our first AI 'Hello World' together!"},
    ],
    temperature=0.001,
    max_tokens=256,
)
print("✅ GB10 Connected:", completion.choices[0].message.content)

✅ GB10 Connected: 🌟 *Beep-boop! Welcome, human friend!* 🌟

I’m your new AI buddy — no cape, no cape needed (though I *do* have a glowing emoji smile 😊). We’re about to do something super cool: say “Hello World”… together! 🌍💻

Whether you’re here to chat, code, or just laugh at my puns (I’ve got a few up my sleeve), I’m thrilled to be your first AI adventure buddy. Let’s make this moment sparkly, silly, and unforgettable!

So… *Hello World!* 🌈  
What’s your favorite thing to say when you start a new project? Or… what’s your favorite snack? (I’m judging you… but kindly.)

Let’s go! 🚀💬


In [2]:
%pip install -q folium streamlit streamlit-folium pillow openai scipy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import folium

from vlm_config import BASE_URL, MODEL
from risk_engine import (
    get_vlm_client,
    get_image_root,
    load_sa_datasets,
    score_image,
    compute_disaster_score,
)

# 1. Verify USAA GB10 VLM connection
client = get_vlm_client()
models = [m.id for m in client.models.list().data]
print(f"✅ Connected to {BASE_URL} | Active Model: {MODEL} | Available: {models}")

# 2. Verify server flood image directory & San Antonio datasets
IMG_ROOT = get_image_root()
all_images = sorted([
    p for p in IMG_ROOT.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"} and ".ipynb_checkpoints" not in str(p)
])
lwc, sso, streetlights = load_sa_datasets()

print(f"✅ Found {len(all_images)} images in: {IMG_ROOT}")
print(f"   Subfolders: {[p.name for p in IMG_ROOT.iterdir() if p.is_dir()]}")
print(f"✅ Loaded SA Layers -> Low Water Crossings: {len(lwc)} | TCEQ Sewer Overflows: {len(sso)} | Smart Streetlights: {len(streetlights)}")

✅ Connected to http://10.246.100.230/v1 | Active Model: qwen3-vl-8b-instruct | Available: ['qwen3-coder-next', 'qwen3.5-122b-a10b', 'qwen3-vl-embedding-8b', 'qwen3-vl-32b-instruct', 'qwen3-vl-8b-instruct']
✅ Found 0 images in: sample_images
   Subfolders: []
✅ Loaded SA Layers -> Low Water Crossings: 4 | TCEQ Sewer Overflows: 3 | Smart Streetlights: 3


In [5]:
from IPython.display import display, Image as IPyImage

if all_images:
    test_img = all_images[0]
    print("Inspecting:", test_img)
    display(IPyImage(filename=str(test_img), width=420))
    
    vlm_out = score_image(str(test_img))
    print(json.dumps(vlm_out, indent=2))

In [6]:
def run_vlm_benchmark(img_root: Path, per_folder: int = 5) -> pd.DataFrame:
    by_folder = {}
    for p in img_root.rglob("*"):
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"} and ".ipynb_checkpoints" not in str(p):
            by_folder.setdefault(p.parent.name, []).append(p)

    rows = []
    for folder, paths in sorted(by_folder.items()):
        for p in sorted(paths)[:per_folder]:
            res = score_image(str(p))
            rows.append({
                "ground_truth_folder": folder,
                "filename": p.name,
                "vlm_status": res["status"],
                "visual_severity": res["visual_severity"],
                "sewage_flag": res["sewage_risk_visual"],
                "summary": res["summary"],
            })
    return pd.DataFrame(rows)


benchmark_df = run_vlm_benchmark(IMG_ROOT, per_folder=5)
display(benchmark_df)

# Plot severity distribution across folders
if not benchmark_df.empty:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for label, grp in benchmark_df.groupby("ground_truth_folder"):
        ax.hist(grp["visual_severity"], bins=10, alpha=0.65, label=f"Folder: {label}")
    ax.set_xlabel("Qwen3-VL Visual Severity Score (0–100)")
    ax.set_ylabel("Number of Images")
    ax.set_title("Dry vs. Flooded Visual Severity Separation on USAA GB10")
    ax.legend()
    plt.tight_layout()
    plt.show()

""


In [7]:
sa_test_locations = [
    {"name": "Alazan Creek @ W Commerce St (High Flood + SSO Zone)", "lat": 29.4275, "lon": -98.5236, "img": str(all_images[0]) if all_images else None},
    {"name": "Brooks Innovation Zone (Active Streetlight Sensor)", "lat": 29.3482, "lon": -98.4351, "img": str(all_images[-1]) if all_images else None},
    {"name": "Medical Center Innovation Zone (Dry Baseline)", "lat": 29.5074, "lon": -98.5752, "img": None},
]

scored_rows = []
for loc in sa_test_locations:
    res = compute_disaster_score(loc["lat"], loc["lon"], loc["img"])
    scored_rows.append({
        "Location": loc["name"],
        "Disaster Probability (%)": res["total_score"],
        "Risk Tier": res["risk_tier"],
        "Sewage Alert": res["sewage_warning"],
        "Nearest LWC": res["nearest_lwc"],
        "Nearest Sensor": res["nearest_sensor"],
        "AI Evidence": res["vlm"]["summary"],
    })

scored_df = pd.DataFrame(scored_rows)
display(scored_df)

,Location,Disaster Probability (%),Risk Tier,Sewage Alert,Nearest LWC,Nearest Sensor,AI Evidence
0,Alazan Creek @ W Commerce St (High Flood + SSO...,39.0,LOW RISK,False,Alazan Creek @ W Commerce St (0m),"Downtown Innovation Zone (0.0 in., 2929m)",No photo supplied; score computed from San Ant...
1,Brooks Innovation Zone (Active Streetlight Sen...,44.0,MODERATE RISK,True,Alazan Creek @ W Commerce St (12299m),"Brooks Innovation Zone (3.4 in., 0m)",No photo supplied; score computed from San Ant...
2,Medical Center Innovation Zone (Dry Baseline),18.0,LOW RISK,False,Leon Creek @ Hausman Rd (6468m),"Medical Center Innovation Zone (0.0 in., 0m)",No photo supplied; score computed from San Ant...


In [8]:
m = folium.Map(location=[29.4241, -98.4936], zoom_start=11, tiles="CartoDB positron")

for _, r in lwc.head(150).iterrows():
    folium.CircleMarker([r["lat"], r["lon"]], radius=4, color="#d9534f", fill=True, popup=f"LWC: {r['label']}").add_to(m)

for _, r in streetlights.head(150).iterrows():
    folium.CircleMarker([r["lat"], r["lon"]], radius=5, color="#0275d8", fill=True, popup=f"Sensor: {r['label']} ({r['flood_level_in']} in.)").add_to(m)

for _, r in sso.head(150).iterrows():
    folium.CircleMarker([r["lat"], r["lon"]], radius=4, color="#8a2be2", fill=True, popup=f"TCEQ SSO: {r['label']}").add_to(m)

for loc, (_, row) in zip(sa_test_locations, scored_df.iterrows()):
    score = row["Disaster Probability (%)"]
    pin_color = "red" if score >= 70 else ("orange" if score >= 40 else "green")
    folium.Marker(
        [loc["lat"], loc["lon"]],
        icon=folium.Icon(color=pin_color, icon="info-sign"),
        popup=f"<b>{loc['name']}</b><br>Score: {score}% ({row['Risk Tier']})<br>Sewage Alert: {row['Sewage Alert']}",
    ).add_to(m)

m

C:\Users\Isaia\anaconda3\envs\gb10\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
